In [ ]:
# Imports & paths

import pandas as pd
import numpy as np
import torch, faiss, pickle
import os
import re
import ast

COVID_TWEETS = "covid_tweet_terms_600.txt"
FAISS = "../UMLS_ENG_CUI_STR_TTY_STY_FAISS.index"
LOOKUP = "../UMLS_ENG_CUI_STR_TTY_STY_LOOKUP.pkl"
SYMPTOM_600_PT_OPENAI = "tweet_terms_PT_openai_4o_600.csv"
SYMPTOM_600_PT_CUI_OPENAI = "tweet_terms_PT_CUI_openai_4o_600.csv"

COVID_TWEETS_INPUT = "covid_tweets_600.csv"
COVID_TWEETS_OUTPUT = "covid_tweets_600_modified.csv"

In [ ]:
# Open AI key

from openai import OpenAI

os.environ["OPENAI_API_KEY"] = OPEN_AI_KEY

client = OpenAI()

In [ ]:
# Load FAISS index + lookup

index = faiss.read_index(FAISS)

with open(LOOKUP, "rb") as f:
    lookup = pickle.load(f)

umls_cuis  = lookup["cuis"]
umls_terms = lookup["terms"]
umls_ttys  = lookup["ttys"]
umls_stys  = lookup["stys"]

print(f"FAISS index loaded with {index.ntotal:,} vectors")

FAISS index loaded with 3,594,105 vectors


In [ ]:
# Load SapBERT for encoding query phrases

from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

C:\ProgramData\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
with open(COVID_TWEETS, "r", encoding="utf-8") as f:
    raw_lines = [line.rstrip("\n") for line in f]

symptom_lines = []
for l in raw_lines:
    l = l.strip()
    if not l:
        symptom_lines.append([])  # preserve line count if there are empty lines
        continue
    symptom_lines.append(ast.literal_eval(l))  # safe parsing

# Flatten to process one symptom at a time, while keeping mapping to restore structure
flat_symptoms = []
idx_map = []  # (line_i, item_j)
for i, lst in enumerate(symptom_lines):
    for j, s in enumerate(lst):
        s = str(s).strip()
        if s == "":
            continue
        flat_symptoms.append(s)
        idx_map.append((i, j))

print("Lines:", len(symptom_lines), "Total symptoms:", len(flat_symptoms))
print("First 2 lines:", symptom_lines[:2])

Lines: 600 Total symptoms: 2125
First 2 lines: [['Cough', 'Sore throat', 'Temp keeps going up and down'], ['headache', 'achy', 'tired', 'Coughing', 'sore throat', 'Fever', 'Hard to breathe', 'tired']]


In [ ]:
# Prompt function

def normalize_symptom(symptom):
    prompt = f"""
    Normalize the following medical symptom into standard medical terminology.
    Provide the single most appropriate, top-level preferred term.
    Respond only with the preferred term.

    Phrase: "{symptom}"
    Preferred term:
    """
    try:
        response = client.responses.create(
            model="gpt-4o",
            input=prompt,
            temperature=0,
        )
        return response.output_text.strip()
    except Exception as e:
        print(f"Error for '{symptom}': {e}")
        return ""

In [ ]:
from tqdm import tqdm

# LLM on every symptom (2125 items) in order
normalized_flat = []
for symptom in tqdm(flat_symptoms, desc="Progress"):
    pt = normalize_symptom(symptom)
    normalized_flat.append(pt)

print("Normalized PTs:", len(normalized_flat))

Progress: 100%|████████████████████████████████████████████████████████████████████| 2125/2125 [23:28<00:00,  1.51it/s]

Normalized PTs: 2125


In [ ]:
# Reconstruct back to 600 lines (same order & grouping)
pt_lines = [[] for _ in range(len(symptom_lines))]
for (line_i, _), pt in zip(idx_map, normalized_flat):
    pt_lines[line_i].append(pt)

# Save as CSV (one row per original line)
df_pt = pd.DataFrame({
    "symptoms": symptom_lines,
    "preferred_terms": pt_lines
})

df_pt.to_csv(SYMPTOM_600_PT_OPENAI, index=False, encoding="utf-8")
print(f"\nSaved preferred terms to {SYMPTOM_600_PT_OPENAI}")
df_pt.head(5)


Saved preferred terms to tweet_terms_PT_openai_4o_600.csv


,symptoms,preferred_terms
0,"[Cough, Sore throat, Temp keeps going up and d...","[Cough, Pharyngitis, Fluctuating temperature]"
1,"[headache, achy, tired, Coughing, sore throat,...","[Cephalalgia, Pain, Fatigue, Cough, Pharyngiti..."
2,"[fever, dry cough, lethargy, fever]","[Pyrexia, Cough, Fatigue, Pyrexia]"
3,"[Nasal congestion, sneezing, slight cough, hea...","[Nasal obstruction, Rhinorrhea, Cough, Cephalgia]"
4,"[fatigued, headache, hot/cold spells, heavy lu...","[Fatigue, Cephalalgia, Temperature dysregulati..."


In [ ]:
def safe_parse_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        return [s]

df_norm = df_pt.copy()

df_norm["preferred_terms"] = df_norm["preferred_terms"].apply(safe_parse_list)

# Flatten (keep original line index)
df_flat = df_norm[["symptoms", "preferred_terms"]].copy()
df_flat["line_i"] = df_flat .index                 # keep mapping to original row
df_flat = df_flat.explode("preferred_terms").dropna()

# Clean PTs
df_flat["preferred_terms"] = df_flat["preferred_terms"].astype(str).str.strip()
df_flat = df_flat[df_flat["preferred_terms"] != ""].reset_index(drop=True)

print("Flat rows (one PT per row):", len(df_flat))
print(df_flat.head())

norm_pts = df_flat["preferred_terms"].tolist()

Flat rows (one PT per row): 2125
                                            symptoms          preferred_terms  \
0  [Cough, Sore throat, Temp keeps going up and d...                    Cough   
1  [Cough, Sore throat, Temp keeps going up and d...              Pharyngitis   
2  [Cough, Sore throat, Temp keeps going up and d...  Fluctuating temperature   
3  [headache, achy, tired, Coughing, sore throat,...              Cephalalgia   
4  [headache, achy, tired, Coughing, sore throat,...                     Pain   

   line_i  
0       0  
1       0  
2       0  
3       1  
4       1  


In [ ]:
# Encode with SapBERT

def encode_queries(texts, batch_size=64, max_length=32):
    embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding phrases"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                        max_length=max_length).to(device)
        with torch.no_grad():
            out = model(**enc)
            pooled = out.last_hidden_state[:, 0, :]           # [CLS]
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        embs.append(pooled.cpu().numpy().astype("float32"))
    return np.vstack(embs)

query_embs = encode_queries(norm_pts)
print("Embeddings shape:", query_embs.shape)

Encoding phrases: 100%|████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 36.08it/s]


Embeddings shape: (2125, 768)


In [ ]:
D, I = index.search(query_embs, k=1)

# Attach predicted CUI back to the flat df
df_flat["predicted_cui"] = [umls_cuis[idx] for idx in I[:, 0]]

# Reconstruct CUI lists per original row
predicted_cuis_lines = [[] for _ in range(len(df_pt))]
for line_i, cui in zip(df_flat["line_i"], df_flat["predicted_cui"]):
    predicted_cuis_lines[int(line_i)].append(cui)

df_pt_cui = pd.DataFrame(
    {
        "symptoms": df_pt["symptoms"].tolist(),
        "preferred_terms": df_pt["preferred_terms"].tolist(),
        "predicted_cuis": predicted_cuis_lines,
    }
)
df_pt_cui.to_csv(SYMPTOM_600_PT_CUI_OPENAI, index=False, encoding="utf-8")
print(f"Saved results to {SYMPTOM_600_PT_CUI_OPENAI}")
print(df_pt_cui.head(5))

Saved results to tweet_terms_PT_CUI_openai_4o_600.csv
                                            symptoms  \
0  [Cough, Sore throat, Temp keeps going up and d...   
1  [headache, achy, tired, Coughing, sore throat,...   
2                [fever, dry cough, lethargy, fever]   
3  [Nasal congestion, sneezing, slight cough, hea...   
4  [fatigued, headache, hot/cold spells, heavy lu...   

                                     preferred_terms  \
0      [Cough, Pharyngitis, Fluctuating temperature]   
1  [Cephalalgia, Pain, Fatigue, Cough, Pharyngiti...   
2                 [Pyrexia, Cough, Fatigue, Pyrexia]   
3  [Nasal obstruction, Rhinorrhea, Cough, Cephalgia]   
4  [Fatigue, Cephalalgia, Temperature dysregulati...   

                                      predicted_cuis  
0                     [C0010200, C0031350, C1504363]  
1  [C0018681, C1549543, C0015672, C0010200, C0031...  
2           [C0015967, C0010200, C0015672, C0015967]  
3           [C0027429, C1260880, C0010200, C0018681]

In [ ]:
df = pd.read_csv(COVID_TWEETS_INPUT, encoding="cp1252")

for col in ["predicted_symptoms", "predicted_preferred_terms", "predicted_cuis"]:
    df[col] = df[col].apply(safe_parse_list)

def find_spans_in_order(text, terms):
    """
    Returns list of (start,end) spans aligned to terms.
    Uses case-insensitive search, left-to-right, one match per term.
    """
    spans = []
    cursor = 0
    for term in terms:
        if not term:
            spans.append(None)
            continue

        # case-insensitive search
        pattern = re.compile(re.escape(term), flags=re.IGNORECASE)
        m = pattern.search(text, pos=cursor)
        if not m:
            spans.append(None)   # couldn’t find; keep alignment
            continue

        spans.append((m.start(), m.end()))
        cursor = m.end()  # move forward so next match is after this one

    return spans

In [ ]:
def replace_using_spans(text, spans, pts, cuis):
    """
    Replaces exact [start:end] with: [PT] (CUI)
    Right-to-left.
    """
    triples = []
    for sp, pt, cui in zip(spans, pts, cuis):
        if sp is None:
            continue
        start, end = sp
        triples.append((start, end, pt, cui))

    # right-to-left
    triples.sort(key=lambda x: x[0], reverse=True)

    out = text
    for start, end, pt, cui in triples:
        out = out[:start] + f"[{pt}] ({cui})" + out[end:]
    return out

In [ ]:
df_out = pd.DataFrame()
df_out["tweet_original"] = df["tweet"]

df_out["tweet_updated"] = [
    replace_using_spans(
        tweet,
        find_spans_in_order(tweet, terms),
        pts,
        cuis
    )
    for tweet, terms, pts, cuis in zip(
        df["tweet"],
        df["predicted_symptoms"],
        df["predicted_preferred_terms"],
        df["predicted_cuis"]
    )
]

df_out.to_csv(COVID_TWEETS_OUTPUT, index=False, encoding="utf-8-sig")
print(df_out.head(3).to_string(index=False))

                                                                                                                                                                                                                                                                                                                                                                              tweet_original                                                                                                                                                                                                                                                                                                                                                                                                                                   tweet_updated
                                                                                                                                     'Self isolation day 1. Cough is productive and nasty. 